# Integrating Synthetic Data into a full ML Ops end-to-end system using SageMaker Pipelines service 



In [ ]:
import boto3
import sagemaker


region = boto3.Session().region_name
role = sagemaker.get_execution_role()
default_bucket = sagemaker.session.Session().default_bucket()

# specify dataset
dataset = 'healthcare-dataset-stroke-data'
# dataset = 'abalone'

# Change these to reflect your project/business name or if you want to separate ModelPackageGroup/Pipeline from the rest of your team
model_package_group_name = f"GretelModelPackageGroup-{dataset}"
pipeline_name = f"GretelPipeline-{dataset}"

### Get the pipeline instance

Here we get the pipeline instance from your pipeline module so that we can work with it.

In [ ]:
from pipelines.abalone.pipeline import get_pipeline
from pipelines.abalone.datasets import datasets

pipeline = get_pipeline(
    region=region,
    dataset=dataset,
    role=role,
    default_bucket=default_bucket,
    model_package_group_name=model_package_group_name,
    pipeline_name=pipeline_name,
)

### Submit the pipeline to SageMaker and start execution

Let's submit our pipeline definition to the workflow service. The role passed in will be used by the workflow service to create all the jobs defined in the steps.

In [ ]:
pipeline.upsert(role_arn=role)

We'll start the pipeline, accepting all the default parameters.

Values can also be passed into these pipeline parameters on starting of the pipeline, and will be covered later. 

In [ ]:
execution = pipeline.start()

### Pipeline Operations: examining and waiting for pipeline execution

Now we describe execution instance and list the steps in the execution to find out more about the execution.

In [ ]:
execution.describe()

We can wait for the execution by invoking `wait()` on the execution:

In [ ]:
execution.wait()

We can list the execution steps to check out the status and artifacts:

In [ ]:
execution.list_steps()

In [ ]:
# clean-up 

client = boto3.client('sagemaker')
pl_list = client.list_pipelines()
for pl in pl_list['PipelineSummaries']:
    print(f"Removing {pl['PipelineName']}")
    client.delete_pipeline(PipelineName=pl['PipelineName'])